# 05 - Model Optimization

In this notebook, we optimize the best machine learning models selected during the regression and classification phases.

The objective is to improve model performance by tuning hyperparameters using techniques such as GridSearchCV and cross-validation.

We also compare optimized models with their initial versions and evaluate their robustness using different performance metrics.

Finally, the best trained models are saved for future use and deployment.

In [13]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.tree import (
    DecisionTreeRegressor,
    DecisionTreeClassifier
)

from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    accuracy_score,
    f1_score,
    classification_report
)

import joblib

In [14]:
#loading the dataset
df = pd.read_csv("../data/cleaned/flight_data.csv")

df.head()

,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
2,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
3,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955
4,Vistara,UK-945,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.33,1,5955


## Regression optimisation

The goal here is to optimize the regression model chosen in 03_regression_models, the Decision Tree Regressor.

In [15]:
# preparing the data

df_sample = df.sample(50000, random_state=42)

X = df_sample.drop("price", axis=1)
y = df_sample["price"]

categorical_cols = X.select_dtypes(include="object").columns
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns 

C:\Users\flori\AppData\Local\Temp\ipykernel_35152\3804400963.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include="object").columns


In [16]:
# preprocessing pipelines for both numeric and categorical data

numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

In [17]:
# splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [18]:
# setting up the pipeline and grid search for regression

pipeline_reg = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(random_state=42))
])

param_grid_reg = {
    "model__max_depth": [5, 10, 20, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

grid_reg = GridSearchCV(
    pipeline_reg,
    param_grid_reg,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_reg.fit(X_train, y_train)

print("Best Parameters:", grid_reg.best_params_)

Best Parameters: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 10}


The optimal Decision Tree Regressor configuration uses no maximum depth restriction, allowing the model to fully capture complex non-linear relationships in the dataset.

The parameter `min_samples_split=10` helps reduce overfitting by requiring a minimum number of samples before creating a new split.

Overall, these hyperparameters improved the model's ability to generalize while maintaining strong predictive performance.

In [19]:
# evaluating the best model on the test set

best_regressor = grid_reg.best_estimator_

y_pred = best_regressor.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R2 Score:", r2)

RMSE: 3453.3818593907345
R2 Score: 0.9771418243636371


The optimized Decision Tree Regressor achieved a strong predictive performance with an RMSE of approximately 3453 and an R² score of 0.977.

This result indicates that the model explains nearly 97.7% of the variance in flight ticket prices, demonstrating its ability to accurately capture complex relationships between features and the target variable.

The relatively low RMSE also suggests that the prediction errors remain limited compared to the overall price range.

In [20]:
# cross-validation scores for the best model

cv_scores = cross_val_score(
    best_regressor,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("Cross-validation scores:", cv_scores)
print("Mean CV Score:", cv_scores.mean())

Cross-validation scores: [0.97714231 0.97836959 0.97694481 0.97660538 0.97795091]
Mean CV Score: 0.97740260087162


The cross-validation results show consistently high R² scores across all folds, with an average score of approximately 0.977.

The low variation between folds indicates that the optimized Decision Tree Regressor is stable and generalizes well to unseen data.

These results confirm the robustness and reliability of the model for flight price prediction.

## Classification optimisation

Now we optimize the classfication model chosen in 04_classification_models, the Decision Tree.

In [21]:
# preparing the data

df["price_class"] = pd.qcut(df["price"], q=3, labels=["low", "medium", "high"])

df_sample = df.sample(50000, random_state=42)

X = df_sample.drop(["price", "price_class"], axis=1)
y = df_sample["price_class"]

categorical_cols = X.select_dtypes(include="object").columns
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns

C:\Users\flori\AppData\Local\Temp\ipykernel_35152\1801032228.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include="object").columns


In [22]:
# preprocessing pipelines for both numeric and categorical data

numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

In [23]:
# splitting the data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [24]:
# setting up the pipeline and grid search for classification

pipeline_clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])

param_grid_clf = {
    "model__max_depth": [5, 10, 20, None],
    "model__min_samples_split": [2, 5, 10],
    "model__criterion": ["gini", "entropy"]
}

grid_clf = GridSearchCV(
    pipeline_clf,
    param_grid_clf,
    cv=5,
    scoring="f1_weighted",
    n_jobs=-1
)

grid_clf.fit(X_train, y_train)

print("Best Parameters:", grid_clf.best_params_)

Best Parameters: {'model__criterion': 'gini', 'model__max_depth': None, 'model__min_samples_split': 2}


The optimal Decision Tree Classifier uses the Gini impurity criterion, with no maximum depth restriction and a minimum of 2 samples required to split a node.

Allowing unlimited depth enables the model to fully capture complex patterns in the dataset, while the Gini criterion provides an efficient measure for node impurity.

Overall, these parameters indicate that the model benefits from high flexibility, which improves its ability to classify flight ticket prices accurately.

In [25]:
# evaluating the best model on the test set

best_classifier = grid_clf.best_estimator_

y_pred = best_classifier.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")

print("Accuracy:", acc)
print("F1-score:", f1)

print(classification_report(y_test, y_pred))

Accuracy: 0.9273
F1-score: 0.927144358012476
              precision    recall  f1-score   support

        high       0.96      0.97      0.97      3310
         low       0.92      0.93      0.92      3345
      medium       0.90      0.88      0.89      3345

    accuracy                           0.93     10000
   macro avg       0.93      0.93      0.93     10000
weighted avg       0.93      0.93      0.93     10000



The optimized Decision Tree Classifier achieved strong performance on the test set, with an accuracy of 0.9273 and a weighted F1-score of approximately 0.9271.

The classification report shows consistently good performance across all classes. The model performs best on the "high" price class, with very high precision and recall, while the "low" and "medium" classes also show balanced and reliable results.

Overall, these results confirm that the model is able to accurately distinguish between different flight price categories, with a good balance between precision and recall across all classes.

In [26]:
# cross-validation scores for the best model
cv_scores = cross_val_score(
    best_classifier,
    X,
    y,
    cv=5,
    scoring="f1_weighted"
)

print("Cross-validation scores:", cv_scores)
print("Mean CV Score:", cv_scores.mean())

Cross-validation scores: [0.92529298 0.92765222 0.92815627 0.92849161 0.92127376]
Mean CV Score: 0.9261733680078186


The cross-validation results show consistently strong performance across all folds, with F1-scores ranging from approximately 0.921 to 0.928.

The small variation between folds indicates that the Decision Tree Classifier is stable and generalizes well to unseen data.

The mean cross-validation score of approximately 0.926 confirms the robustness of the model and its ability to maintain high predictive performance across different subsets of the dataset.

## Conclusion

The optimized models slightly improved performance compared to the initial baseline models.

Both regression and classification tasks demonstrate that Decision Tree-based models are highly effective for this dataset, due to their ability to capture complex non-linear relationships between features and the target variable.